# smoke

> Compile the tiny wav2vec 2.0 graph and check that a forward pass returns finite logits.


In [ ]:
#| default_exp smoke


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
from __future__ import annotations


In [ ]:
#| export
import numpy as np

from nada.config import tiny_config
from nada.features import ctc_decode
from nada.model import build, compile_model, to_device


In [ ]:
#| export
def main() -> None:
    config = tiny_config()
    model = build(config)
    batch, samples = 2, 400
    frames = config.output_length(samples)
    compiled = compile_model(model, batch=batch, num_samples=samples)
    waveform = np.random.default_rng(0).standard_normal((batch, samples)).astype(np.float32)
    logits, hidden = compiled(to_device(model, waveform))
    logits_np = np.asarray(logits.to_numpy())
    hidden_np = np.asarray(hidden.to_numpy())
    if logits_np.shape != (batch, frames, config.vocab_size):
        raise SystemExit(f"logits shape {logits_np.shape}")
    if hidden_np.shape != (batch, frames, config.hidden_size):
        raise SystemExit(f"hidden shape {hidden_np.shape}")
    if not np.isfinite(logits_np).all() or not np.isfinite(hidden_np).all():
        raise SystemExit("outputs are not finite")
    vocab = {index: f"<{index}>" for index in range(config.vocab_size)}
    ctc_decode(logits_np.argmax(axis=-1), vocab)
    print("smoke ok", "logits", tuple(logits_np.shape), "hidden", tuple(hidden_np.shape))


In [ ]:
#| export
if __name__ == "__main__":
    main()
